In [4]:
import pandas as pd

# Loading data

In [16]:
data_file = "VAL_THO_GF.xlsx"

# excel column to English mapping
cols = {
    'Fachabteilung': 'dept',
    'Saal': 'room',
    'Notfallkategorie': 'priority',
    'procedureData_originductionmethod': 'anes-method',

    'In Schleuse': 'arrival-airlock',
    'In Einleitung': 'arrival-induction',
    'documented_anesthesiastarttimestamp': 'anes-start',
    'documented_anesthesiareleasetimestamp': 'anes-release',
    'documented_incisiontimestamp': 'incision',
    'auxilaryData_anesthesiaStartTimestamp': 'planned-anes-start',
    'auxilaryData_anesthesiaReleaseTimestamp': 'planned-anes-release',
    'auxilaryData_incisionTimestamp': 'planned-incision',
    'auxilaryData_expectedAnesthesiaReleaseTimestamp': 'expected-anes-release',
}

# columns with time information
time_cols = ['arrival-airlock', 'arrival-induction', 'anes-start', 'anes-release', 'incision', 'planned-anes-start', 'planned-anes-release', 'planned-incision', 'expected-anes-release']

# loading data
df = pd.read_excel("data/" + data_file, header=0, usecols=cols.keys(), dtype=str)
df = df.rename(columns=cols)

# converting time columns to datetime
for col in time_cols:
    if col in df.columns:
        col_data = df[col]
        if col_data.str.contains(r'\d{4}-\d{2}-\d{2}').any():
            timestamps = pd.to_datetime(col_data, format='%Y-%m-%d %H:%M:%S', errors='coerce')
        elif col_data.str.contains(r'\d{2}\.\d{2}\.\d{4}').any():
            timestamps = pd.to_datetime(col_data, dayfirst=True, errors='coerce')
        df[col] = timestamps

# calculating wait time for anesthesia
df['best-arrival'] = df['arrival-induction'].fillna(df['arrival-airlock'])
df['wait_for_anes_min'] = (df['anes-start'] - df['best-arrival']).dt.total_seconds() / 60
df.loc[df['wait_for_anes_min'] < 0, 'wait_for_anes_min'] = 0

# calculating schedule delays
df['delay_start_min'] = (df['anes-start'] - df['planned-anes-start']).dt.total_seconds() / 60
df['delay_release_min'] = (df['anes-release'] - df['planned-anes-release']).dt.total_seconds() / 60
df['delay_incision_min'] = (df['incision'] - df['planned-incision']).dt.total_seconds() / 60
df['delay_release_exp_min'] = (df['anes-release'] - df['expected-anes-release']).dt.total_seconds() / 60

print("--- Average Logistics Metrics ---")
print(f"Average Physical Wait for Anesthesia: {df['wait_for_anes_min'].mean():.1f} mins")
print(f"Average Delay for Anesthesia Start:   {df['delay_start_min'].mean():.1f} mins")
print(f"Average Delay for Anesthesia Release: {df['delay_release_min'].mean():.1f} mins")
print(f"Average Delay for Incision (Surgery): {df['delay_incision_min'].mean():.1f} mins")
print(f"Average Delay for Anesthesia Release from expected value: {df['delay_release_exp_min'].mean():.1f} mins")

--- Average Logistics Metrics ---
Average Physical Wait for Anesthesia: 11.7 mins
Average Delay for Anesthesia Start:   0.0 mins
Average Delay for Anesthesia Release: 0.0 mins
Average Delay for Incision (Surgery): 0.1 mins
Average Delay for Anesthesia Release from expected value: 1.5 mins


# Split data based on clusters

In [ ]:
clusters: [['OP2']]